In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Reading the BF data

In [3]:
bf_path = '/content/drive/MyDrive/ALL_CLEAN_DEIDEN_NAME AND ECMO DATA(Sheet1) (1) (version 2).csv'
cols = ['ID','age_days', 'Diagnosis', 'sex (1:M, 2:F)']
df = pd.read_csv(bf_path, usecols=cols)

s = pd.to_numeric(df['age_days'], errors='coerce')
df['age_days'] = pd.cut(s, [0,10,100,1000,np.inf], labels=[0,1,2,3],
                        include_lowest=True).astype('Int64')

for col in ['age_days']:
    s = pd.to_numeric(df[col], errors='coerce')
    mx = s.max(skipna=True)
    if pd.notna(mx) and mx != 0:
        df[col] = s / (mx*2)
    else:
        df[col] = s
print(df.head())

   ID  sex (1:M, 2:F)  age_days            Diagnosis
0   1               1  0.333333       Cardiac Arrest
1   2               1  0.333333       Cardiac Arrest
2   3               2       0.5               Sepsis
3   4               1       0.0  Respiratory Failure
4   5               2       0.0  Respiratory Failure


In [4]:
diag_clean = (
    df['Diagnosis']
      .astype('string')
      .str.strip()
      .str.replace(r'\s+', ' ', regex=True)
      .fillna('Unknown')
)

codes, uniques = pd.factorize(diag_clean, sort=True)
df['Diagnosis'] = codes.astype('int64')
diagnosis_mapping = {cat: int(i) for i, cat in enumerate(uniques)}
print("Diagnosis mapping (category -> code):", diagnosis_mapping)
# Peek
print(df.head())

Diagnosis mapping (category -> code): {'Cardiac Arrest': 0, 'Cardiogenic Shock': 1, 'Respiratory Failure': 2, 'Sepsis': 3, 'Septic shock': 4}
   ID  sex (1:M, 2:F)  age_days  Diagnosis
0   1               1  0.333333          0
1   2               1  0.333333          0
2   3               2       0.5          3
3   4               1       0.0          2
4   5               2       0.0          2


In [5]:
df.shape

(72, 4)

### No normalization!

In [12]:
# ===== Tab-only XGBoost with Youden-J (computed on TEST) & fixed-0.5 metrics =====
from pathlib import Path
import re, random, math, os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, confusion_matrix
)
import matplotlib.pyplot as plt

# --- XGBoost (install if missing) ---
try:
    from xgboost import XGBClassifier
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBClassifier

# =========================
# Config
# =========================
SPLIT_DIR = r"/content/drive/MyDrive/CD/patient_data_clean_1800s_nozero_181920212223_in3days_v2"
POS_PATIENTS = {1, 2, 16, 19, 21, 22, 25, 37, 39, 43, 44, 47, 50, 56, 58, 62, 65, 66, 73, 78}

SEED      = 1
K_FOLDS   = 7
SAVE_ROC  = True  # save a ROC curve per fold (test set)

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================
# Helpers
# =========================
PATIENT_NUM_RX = re.compile(r'^ID(\d+)')

def patient_num_from_path(p):
    m = PATIENT_NUM_RX.match(Path(p).stem)
    return int(m.group(1)) if m else None

def label_for_file(p: Path) -> int:
    pid = patient_num_from_path(p)
    return 1 if (pid is not None and pid in POS_PATIENTS) else 0

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

def mean_std(a):
    a = np.asarray(a, float)
    return float(np.nanmean(a)), float(np.nanstd(a))

def youden_threshold_from_probs(y_true, y_prob):
    """Compute Youden-J threshold directly on the given set (here: TEST)."""
    if len(np.unique(y_true)) < 2:
        return 0.5, 0.0, 0.0, 0.0
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    j = tpr - fpr
    i = int(np.argmax(j))
    return float(thr[i]), float(j[i]), float(fpr[i]), float(tpr[i])

def metrics_at_threshold(y_true, y_prob, thr):
    """ACC, PREC, REC, SPEC, F1 at threshold thr."""
    y_pred = (y_prob >= thr).astype(int)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)   # sensitivity
    spec = specificity_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    return acc, prec, rec, spec, f1

def files_to_arrays(files, FEAT_DIM, ID_TO_FEAT):
    X = np.empty((len(files), FEAT_DIM), dtype=np.float32)
    y = np.empty((len(files),), dtype=np.int32)
    for i, f in enumerate(files):
        pid = patient_num_from_path(f)
        X[i] = ID_TO_FEAT[pid]
        y[i] = label_for_file(f)
    return X, y

def _fmt_list(name, arr):
    print(f"{name:>12}:", [None if (x is None or (isinstance(x,float) and np.isnan(x)))
                           else round(float(x),4) for x in arr])

# =========================
# Load diagnostic feature table from `df` (must exist in memory)
# Columns: 'ID' + feature columns
# =========================
feats_df = df.copy()  # <-- your in-memory DataFrame
if "ID" not in feats_df.columns:
    raise RuntimeError("Your features DataFrame must contain column 'ID'.")

FEAT_COLS = [c for c in feats_df.columns if c != "ID"]
feats_df["ID"] = pd.to_numeric(feats_df["ID"], errors="coerce").astype("Int64")
feats_df = feats_df.dropna(subset=["ID"] + FEAT_COLS).copy()
feats_df["ID"] = feats_df["ID"].astype(int)

ID_TO_FEAT = {
    int(r["ID"]): r[FEAT_COLS].astype("float32").to_numpy()
    for _, r in feats_df.iterrows()
}
FEAT_DIM = len(FEAT_COLS)
print("FEAT_DIM =", FEAT_DIM)

# =========================
# Enumerate files and valid IDs
# =========================
split_dir = Path(SPLIT_DIR)
all_csvs = sorted(split_dir.glob("*.csv"))
if not all_csvs:
    raise FileNotFoundError(f"No CSV found in {SPLIT_DIR}")

id_to_files = {}
for f in all_csvs:
    pid = patient_num_from_path(f)
    if pid is None:
        continue
    id_to_files.setdefault(pid, []).append(f)

all_ids   = sorted(id_to_files.keys())
valid_ids = [pid for pid in all_ids if pid in ID_TO_FEAT]
if not valid_ids:
    raise RuntimeError("No overlapping patient IDs between files and feature table.")
labels_all = np.array([1 if pid in POS_PATIENTS else 0 for pid in valid_ids], dtype=int)

print("Total valid IDs:", len(valid_ids),
      "| Pos IDs:", labels_all.sum(),
      "| Neg IDs:", (1 - labels_all).sum())

# =========================
# CV with XGBoost
# =========================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

# Per-fold TEST metrics @ fixed 0.5
AUC_list, ACC_list, PREC_list, REC_list, SPEC_list, F1_list = [], [], [], [], [], []
# Per-fold TEST metrics @ Youden-J (computed on TEST)
AUC_J_list, ACC_J_list, PREC_J_list, REC_J_list, SPEC_J_list, F1_J_list = [], [], [], [], [], []

fold_rows = []

for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(valid_ids, labels_all), start=1):
    ids_train_full = [valid_ids[i] for i in tr_idx]
    ids_test       = [valid_ids[i] for i in te_idx]

    # Take a small validation set from training IDs (only for training monitor; not for thresholding)
    train_labels_full = np.array([1 if pid in POS_PATIENTS else 0 for pid in ids_train_full], dtype=int)
    ids_tr, ids_val = train_test_split(
        ids_train_full, test_size=0.10, random_state=SEED, stratify=train_labels_full
    )

    train_files = [f for pid in ids_tr  for f in id_to_files[pid]]
    val_files   = [f for pid in ids_val for f in id_to_files[pid]]
    test_files  = [f for pid in ids_test for f in id_to_files[pid]]

    # Arrays
    X_tr, y_tr = files_to_arrays(train_files, FEAT_DIM, ID_TO_FEAT)
    X_va, y_va = files_to_arrays(val_files,   FEAT_DIM, ID_TO_FEAT)
    X_te, y_te = files_to_arrays(test_files,  FEAT_DIM, ID_TO_FEAT)

    # Class imbalance weighting
    pos_ratio = (y_tr == 1).sum()
    neg_ratio = (y_tr == 0).sum()
    spw = (neg_ratio / max(pos_ratio, 1))

    # Model
    model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=10,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=spw,
        random_state=SEED,
        n_jobs=-1,
    )
    # (VAL used only for monitoring)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    # ---- TEST predictions ----
    prob_te = model.predict_proba(X_te)[:, 1]

    # AUC (threshold-free)
    auc = roc_auc_score(y_te, prob_te) if len(np.unique(y_te)) == 2 else np.nan

    # ---- Thresholds ----
    # (1) Fixed 0.5
    acc_05, prec_05, rec_05, spec_05, f1_05 = metrics_at_threshold(y_te, prob_te, thr=0.5)

    # (2) Youden-J computed ON TEST (per your request)
    t_star_test, J_test, fpr_t, tpr_t = youden_threshold_from_probs(y_te, prob_te)
    acc_J, prec_J, rec_J, spec_J, f1_J = metrics_at_threshold(y_te, prob_te, thr=t_star_test)

    # Collect lists
    AUC_list.append(auc); ACC_list.append(acc_05); PREC_list.append(prec_05)
    REC_list.append(rec_05); SPEC_list.append(spec_05); F1_list.append(f1_05)

    AUC_J_list.append(auc)  # same AUC across thresholds
    ACC_J_list.append(acc_J);   PREC_J_list.append(prec_J)
    REC_J_list.append(rec_J);   SPEC_J_list.append(spec_J);   F1_J_list.append(f1_J)

    # Record fold summary row
    fold_rows.append({
        "fold": fold_idx, "n_test": len(y_te),
        "youden_t_test": t_star_test, "youden_J_test": J_test,
        "AUC": auc,
        "ACC@0.5": acc_05, "Precision@0.5": prec_05, "Recall@0.5": rec_05, "Specificity@0.5": spec_05, "F1@0.5": f1_05,
        "ACC@J":  acc_J,   "Precision@J":  prec_J,  "Recall@J":  rec_J,  "Specificity@J":  spec_J,  "F1@J":  f1_J
    })

    # Optional ROC
    if SAVE_ROC and len(np.unique(y_te))==2:
        fpr, tpr, _ = roc_curve(y_te, prob_te)
        plt.figure()
        plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
        plt.plot([0,1],[0,1], linestyle="--", linewidth=1)
        plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title(f"ROC — TEST Fold {fold_idx:02d} (n={len(y_te)})")
        plt.legend(loc="lower right"); plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"roc_xgb_test_fold_{fold_idx:02d}.png", dpi=200)
        plt.close()

# =========================
# Report & save
# =========================
print("\n=== TEST metrics per fold — threshold 0.5 ===")
_fmt_list("AUC",  AUC_list)
_fmt_list("ACC",  ACC_list)
_fmt_list("PREC", PREC_list)
_fmt_list("REC",  REC_list)
_fmt_list("SPEC", SPEC_list)
_fmt_list("F1",   F1_list)

print("\n=== TEST metrics per fold — Youden-J (computed on TEST) ===")
_fmt_list("AUC@J",   AUC_J_list)
_fmt_list("ACC@J",   ACC_J_list)
_fmt_list("PREC@J",  PREC_J_list)
_fmt_list("REC@J",   REC_J_list)
_fmt_list("SPEC@J",  SPEC_J_list)
_fmt_list("F1@J",    F1_J_list)

print("\n=== Mean ± SD across folds (TEST) — 0.5 ===")
for nm, arr in [("AUC",AUC_list), ("ACC",ACC_list), ("Precision",PREC_list),
                ("Recall",REC_list), ("Specificity",SPEC_list), ("F1",F1_list)]:
    m,s = mean_std(arr); print(f"{nm:>12}: {m:.4f} ± {s:.4f}")

print("\n=== Mean ± SD across folds (TEST) — Youden-J (on TEST) ===")
for nm, arr in [("AUC@J",AUC_J_list), ("ACC@J",ACC_J_list), ("Precision@J",PREC_J_list),
                ("Recall@J",REC_J_list), ("Specificity@J",SPEC_J_list), ("F1@J",F1_J_list)]:
    m,s = mean_std(arr); print(f"{nm:>12}: {m:.4f} ± {s:.4f}")

pd.DataFrame(fold_rows).to_csv("xgb_tabonly_test_metrics_by_fold_with_youden_TEST.csv", index=False)
print("\nSaved: xgb_tabonly_test_metrics_by_fold_with_youden_TEST.csv")

# Expose lists for reuse
TEST_AUC_LIST        = AUC_list
TEST_ACC_LIST        = ACC_list
TEST_PREC_LIST       = PREC_list
TEST_REC_LIST        = REC_list
TEST_SPEC_LIST       = SPEC_list
TEST_F1_LIST         = F1_list

TEST_AUC_J_LIST      = AUC_J_list
TEST_ACC_J_LIST      = ACC_J_list
TEST_PREC_J_LIST     = PREC_J_list
TEST_REC_J_LIST      = REC_J_list
TEST_SPEC_J_LIST     = SPEC_J_list
TEST_F1_J_LIST       = F1_J_list


FEAT_DIM = 3
Total valid IDs: 42 | Pos IDs: 13 | Neg IDs: 29

=== TEST metrics per fold — threshold 0.5 ===
         AUC: [0.9618, 1.0, 0.5507, 0.0322, 0.95, 1.0, 1.0]
         ACC: [0.9695, 0.9911, 0.3768, 0.533, 0.9286, 0.758, 0.1203]
        PREC: [1.0, 0.9783, 0.0, 0.0, 0.7037, 1.0, 0.0714]
         REC: [0.9111, 1.0, 0.0, 0.0, 0.95, 0.6082, 1.0]
        SPEC: [1.0, 0.9851, 0.7536, 0.8582, 0.9245, 1.0, 0.0565]
          F1: [0.9535, 0.989, 0.0, 0.0, 0.8085, 0.7564, 0.1333]

=== TEST metrics per fold — Youden-J (computed on TEST) ===
       AUC@J: [0.9618, 1.0, 0.5507, 0.0322, 0.95, 1.0, 1.0]
       ACC@J: [0.9695, 1.0, 0.7754, 0.6211, 0.9921, 1.0, 1.0]
      PREC@J: [1.0, 1.0, 0.69, 0.0, 1.0, 1.0, 1.0]
       REC@J: [0.9111, 1.0, 1.0, 0.0, 0.95, 1.0, 1.0]
      SPEC@J: [1.0, 1.0, 0.5507, 1.0, 1.0, 1.0, 1.0]
        F1@J: [0.9535, 1.0, 0.8166, 0.0, 0.9744, 1.0, 1.0]

=== Mean ± SD across folds (TEST) — 0.5 ===
         AUC: 0.7849 ± 0.3420
         ACC: 0.6682 ± 0.3104
   Precision:

In [11]:
# --- Print all metric lists (per-fold) ---
def _fmt(x):
    return "None" if (x is None or (isinstance(x,float) and np.isnan(x))) else f"{x:.4f}"

print("\n=== Per-fold TEST metric lists ===")
print("AUC :", [ _fmt(x) for x in TEST_AUC_LIST ])
print("ACC :", [ _fmt(x) for x in TEST_ACC_LIST ])
print("PREC:", [ _fmt(x) for x in TEST_PREC_LIST ])
print("REC :", [ _fmt(x) for x in TEST_REC_LIST ])
print("SPEC:", [ _fmt(x) for x in TEST_SPEC_LIST ])
print("F1  :", [ _fmt(x) for x in TEST_F1_LIST ])

# --- Also print mean ± SD for each list ---
def mean_std_safe(arr):
    a = np.array([x for x in arr if not (isinstance(x,float) and np.isnan(x))], dtype=float)
    m = float(a.mean()) if len(a) else float("nan")
    s = float(a.std()) if len(a) else float("nan")
    return m, s

for name, arr in [
    ("AUC",  TEST_AUC_LIST),
    ("ACC",  TEST_ACC_LIST),
    ("PREC", TEST_PREC_LIST),
    ("REC",  TEST_REC_LIST),
    ("SPEC", TEST_SPEC_LIST),
    ("F1",   TEST_F1_LIST),
]:
    m, s = mean_std_safe(arr)
    print(f"{name:>6} mean±SD: {m:.4f} ± {s:.4f}")

# --- (Optional) save all lists to disk ---
all_lists = {
    "AUC":  TEST_AUC_LIST,
    "ACC":  TEST_ACC_LIST,
    "PREC": TEST_PREC_LIST,
    "REC":  TEST_REC_LIST,
    "SPEC": TEST_SPEC_LIST,
    "F1":   TEST_F1_LIST,
}
pd.DataFrame(all_lists).to_csv("xgb_tabonly_test_metric_lists.csv", index=False)
pd.Series(all_lists).to_json("xgb_tabonly_test_metric_lists.json")
print("\nSaved lists to xgb_tabonly_test_metric_lists.csv and .json")


=== Per-fold TEST metric lists ===
AUC : ['0.9618', '1.0000', '0.5507', '0.0322', '0.9500', '1.0000', '1.0000']
ACC : ['0.9695', '0.9911', '0.3768', '0.5330', '0.9286', '0.7580', '0.1203']
PREC: ['1.0000', '0.9783', '0.0000', '0.0000', '0.7037', '1.0000', '0.0714']
REC : ['0.9111', '1.0000', '0.0000', '0.0000', '0.9500', '0.6082', '1.0000']
SPEC: ['1.0000', '0.9851', '0.7536', '0.8582', '0.9245', '1.0000', '0.0565']
F1  : ['0.9535', '0.9890', '0.0000', '0.0000', '0.8085', '0.7564', '0.1333']
   AUC mean±SD: 0.7849 ± 0.3420
   ACC mean±SD: 0.6682 ± 0.3104
  PREC mean±SD: 0.5362 ± 0.4543
   REC mean±SD: 0.6385 ± 0.4224
  SPEC mean±SD: 0.7968 ± 0.3135
    F1 mean±SD: 0.5201 ± 0.4204

Saved lists to xgb_tabonly_test_metric_lists.csv and .json


In [13]:
# --- Print all metric lists (per-fold) ---
def _fmt(x):
    return "None" if (x is None or (isinstance(x,float) and np.isnan(x))) else f"{x:.4f}"

print("\n=== Per-fold TEST metric lists ===")
print("AUC :", [ _fmt(x) for x in TEST_AUC_J_LIST ])
print("ACC :", [ _fmt(x) for x in TEST_ACC_J_LIST ])
print("PREC:", [ _fmt(x) for x in TEST_PREC_J_LIST ])
print("REC :", [ _fmt(x) for x in TEST_REC_J_LIST ])
print("SPEC:", [ _fmt(x) for x in TEST_SPEC_J_LIST ])
print("F1  :", [ _fmt(x) for x in TEST_F1_J_LIST ])

# --- Also print mean ± SD for each list ---
def mean_std_safe(arr):
    a = np.array([x for x in arr if not (isinstance(x,float) and np.isnan(x))], dtype=float)
    m = float(a.mean()) if len(a) else float("nan")
    s = float(a.std()) if len(a) else float("nan")
    return m, s

for name, arr in [
    ("AUC",  TEST_AUC_LIST),
    ("ACC",  TEST_ACC_LIST),
    ("PREC", TEST_PREC_LIST),
    ("REC",  TEST_REC_LIST),
    ("SPEC", TEST_SPEC_LIST),
    ("F1",   TEST_F1_LIST),
]:
    m, s = mean_std_safe(arr)
    print(f"{name:>6} mean±SD: {m:.4f} ± {s:.4f}")

# --- (Optional) save all lists to disk ---
all_lists = {
    "AUC":  TEST_AUC_LIST,
    "ACC":  TEST_ACC_LIST,
    "PREC": TEST_PREC_LIST,
    "REC":  TEST_REC_LIST,
    "SPEC": TEST_SPEC_LIST,
    "F1":   TEST_F1_LIST,
}
pd.DataFrame(all_lists).to_csv("xgb_tabonly_test_metric_lists.csv", index=False)
pd.Series(all_lists).to_json("xgb_tabonly_test_metric_lists.json")
print("\nSaved lists to xgb_tabonly_test_metric_lists.csv and .json")


=== Per-fold TEST metric lists ===
AUC : ['0.9618', '1.0000', '0.5507', '0.0322', '0.9500', '1.0000', '1.0000']
ACC : ['0.9695', '1.0000', '0.7754', '0.6211', '0.9921', '1.0000', '1.0000']
PREC: ['1.0000', '1.0000', '0.6900', '0.0000', '1.0000', '1.0000', '1.0000']
REC : ['0.9111', '1.0000', '1.0000', '0.0000', '0.9500', '1.0000', '1.0000']
SPEC: ['1.0000', '1.0000', '0.5507', '1.0000', '1.0000', '1.0000', '1.0000']
F1  : ['0.9535', '1.0000', '0.8166', '0.0000', '0.9744', '1.0000', '1.0000']
   AUC mean±SD: 0.7849 ± 0.3420
   ACC mean±SD: 0.6682 ± 0.3104
  PREC mean±SD: 0.5362 ± 0.4543
   REC mean±SD: 0.6385 ± 0.4224
  SPEC mean±SD: 0.7968 ± 0.3135
    F1 mean±SD: 0.5201 ± 0.4204

Saved lists to xgb_tabonly_test_metric_lists.csv and .json
